# TFLite export, export fidelity and CPU latency: adopted model vs YOLOv12n

Both models are exported to FP32 TFLite in the **same
session, with the same Ultralytics version and the same settings** as the published adopted export, then:

1. export fidelity on the fold-1 validation images (PyTorch vs TFLite, same validator settings);
2. single-image end-to-end CPU latency (image loading, preprocessing, inference, NMS) and memory, 4 interpreter
   threads, 10 warm-up inferences, 100 fold-1 images stratified by class, 3 rounds in alternating order;
3. a package for the smartphone benchmark (the TFLite files and the same 100 images).

**Kaggle settings:** Accelerator **None** (CPU only, no GPU quota), **Internet on**. Inputs: the dataset
`figshare-yolo-trainval-official` and a dataset holding the three published files
(`best.pt`, `bestYOLO.pt`, `tumor_detector.tflite`), checked by SHA-256.
Run with *Save Version → Save & Run All (Commit)* to keep a versioned record (about 30–40 min).

## 1. Configuration

In [ ]:
ULTRALYTICS_VERSION = "8.4.104"   # version of the published adopted export (embedded in its metadata)
EXPORT_ARGS = dict(format="litert", imgsz=640, batch=1)  # 'tflite' is an alias of 'litert' in 8.4.104
N_IMAGES, N_WARMUP, N_ROUNDS, THREADS, SEED = 100, 10, 3, 4, 42
PREDICT = dict(imgsz=640, conf=0.25, iou=0.7, max_det=300)  # Ultralytics predict defaults, stated explicitly
GITHUB_CKPT = ("https://github.com/Reflet-hub/lightweight-yolov12-brain-tumor-detection"
               "/raw/main/models/adopted_model_best.pt")

from pathlib import Path
KAGGLE_INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")

## 2. Environment

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", f"ultralytics=={ULTRALYTICS_VERSION}"])
# The LiteRT converter (litert-torch, ai-edge-litert) is installed by Ultralytics at export time,
# exactly as for the published adopted export; its versions are recorded after the export.

import hashlib, json, os, platform, shutil, time, zipfile
from importlib.metadata import version
import numpy as np, pandas as pd, torch, ultralytics, yaml
from ultralytics import YOLO

assert ultralytics.__version__ == ULTRALYTICS_VERSION, ultralytics.__version__
cpu_model = next((l.split(":", 1)[1].strip() for l in open("/proc/cpuinfo") if l.startswith("model name")), "?")
ENV = {"python": platform.python_version(), "torch": torch.__version__, "ultralytics": ultralytics.__version__,
       "cpu": cpu_model, "cpu_count": os.cpu_count(), "gpu": torch.cuda.is_available()}
print(json.dumps(ENV, indent=2))

## 3. Inputs and checkpoint provenance

In [ ]:
# The three published files, identified by their SHA-256.
EXPECTED = {
    "best.pt": "8b5cd225a8367b08431ff3d7120c2e2aa750470160da66ca2b808ae2ef457bea",               # adopted model
    "bestYOLO.pt": "b30d2637d5f7a10e448eae58af681f086d2dd090d977b06ed9c1dbdbbab9fc68",           # YOLOv12n baseline
    "tumor_detector.tflite": "f9824e0751c2ba2421ab926872bad1448695226ba27ae029d5faf00c495de1af",  # adopted export, 2026-08-07
}


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


DATA = next(KAGGLE_INPUT.rglob("split_metadata.csv")).parent
INPUTS = next(KAGGLE_INPUT.rglob("bestYOLO.pt")).parent
for name, digest in EXPECTED.items():
    assert sha256(INPUTS / name) == digest, f"{name}: not the file of the article (SHA-256 mismatch)"
github_pt = WORK / "adopted_model_best_github.pt"
subprocess.check_call(["wget", "-q", "-O", str(github_pt), GITHUB_CKPT])
assert sha256(github_pt) == EXPECTED["best.pt"], "The adopted checkpoint differs from the one published on GitHub"
print("Inputs verified (SHA-256); the adopted checkpoint is identical to the one published on GitHub.")


def describe(path):
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    targs = ckpt.get("train_args", {})
    tr = {k.strip(): v for k, v in (ckpt.get("train_results") or {}).items()}
    fit = tr["metrics/mAP50-95(B)"]
    best = int(np.argmax(fit))
    return {"file": path.name, "saved": ckpt.get("date"), "ultralytics": ckpt.get("version"),
            "params": sum(p.numel() for p in ckpt["model"].parameters()),
            **{k: targs.get(k) for k in ("model", "seed", "epochs", "patience", "batch", "cos_lr", "freeze")},
            "epochs_run": int(tr["epoch"][-1]), "best_epoch": int(tr["epoch"][best]),
            "best_val_mAP50": round(float(tr["metrics/mAP50(B)"][best]), 4),
            "best_val_mAP50_95": round(float(fit[best]), 4)}


d_adopted, d_yolo = describe(INPUTS / "best.pt"), describe(INPUTS / "bestYOLO.pt")
print(pd.DataFrame([d_adopted, d_yolo]).T.to_string())
# Values reported in the article
assert (d_adopted["seed"], d_adopted["best_epoch"], d_adopted["epochs_run"], d_adopted["cos_lr"]) == (42, 103, 203, True)
assert (d_yolo["seed"], d_yolo["best_epoch"], d_yolo["epochs_run"], d_yolo["cos_lr"]) == (42, 155, 255, True)

## 4. Export both models to FP32 TFLite (same session, same settings)

In [ ]:
EXPORT = WORK / "export"
MODELS = {}
for name, src in (("adopted", INPUTS / "best.pt"), ("yolov12n", INPUTS / "bestYOLO.pt")):
    folder = EXPORT / name
    folder.mkdir(parents=True, exist_ok=True)
    pt = folder / f"{name}.pt"
    shutil.copy2(src, pt)
    tflite = Path(YOLO(str(pt), task="detect").export(**EXPORT_ARGS))
    MODELS[f"{name}_pt"] = pt
    MODELS[f"{name}_tflite"] = tflite
for pkg in ("litert-torch", "ai-edge-litert", "torch"):
    ENV[pkg] = version(pkg)
print({k: ENV[k] for k in ("litert-torch", "ai-edge-litert", "torch")})
aug07 = EXPORT / "adopted_aug07.tflite"  # export of the article (2026-08-07)
shutil.copy2(INPUTS / "tumor_detector.tflite", aug07)
MODELS["adopted_aug07_tflite"] = aug07


def tflite_metadata(path):
    with zipfile.ZipFile(path) as zf:
        return json.loads(zf.read("metadata.json"))


rows = []
for key, path in MODELS.items():
    row = {"model": key, "file": path.name, "size_MiB": round(path.stat().st_size / 2**20, 3), "sha256": sha256(path)[:16]}
    if path.suffix == ".tflite":
        meta = tflite_metadata(path)
        row.update(ultralytics=meta.get("version"), exported=meta.get("date"), imgsz=meta.get("imgsz"), args=meta.get("args"))
    rows.append(row)
EXPORTS = pd.DataFrame(rows)
print(EXPORTS.to_string(index=False))

## 5. Export fidelity on the fold-1 validation images

In [ ]:
data_yaml = WORK / "data_fold1.yaml"
data_yaml.write_text(yaml.safe_dump({"path": str(DATA), "train": "train/images", "val": "val/images", "nc": 3,
                                     "names": {0: "meningioma", 1: "glioma", 2: "pituitary_tumor"}}, sort_keys=False))
fidelity = []
for key in ("adopted_pt", "adopted_tflite", "adopted_aug07_tflite", "yolov12n_pt", "yolov12n_tflite"):
    m = YOLO(str(MODELS[key]), task="detect").val(
        data=str(data_yaml), split="val", imgsz=640, batch=1, conf=0.001, iou=0.7, device="cpu",
        plots=False, verbose=False, project=str(WORK / "val"), name=key, exist_ok=True)
    row = {"model": key, "P": m.box.mp, "R": m.box.mr, "mAP50": m.box.map50, "mAP50_95": m.box.map}
    for i, c in enumerate(m.box.ap_class_index):
        row[f"AP50_{m.names[int(c)]}"] = float(m.box.ap50[i])
    fidelity.append(row)
FIDELITY = pd.DataFrame(fidelity).set_index("model")
pairs = [("adopted_tflite", "adopted_pt"), ("adopted_aug07_tflite", "adopted_pt"), ("yolov12n_tflite", "yolov12n_pt")]
for tfl, pt in pairs:
    d = FIDELITY.loc[tfl, ["mAP50", "mAP50_95"]] - FIDELITY.loc[pt, ["mAP50", "mAP50_95"]]
    print(f"{tfl} - {pt}: ΔmAP50 = {d['mAP50']:+.4f}, ΔmAP50-95 = {d['mAP50_95']:+.4f}")
print(FIDELITY.round(4).to_string())

## 6. Benchmark images: 100 fold-1 images, stratified by class

In [ ]:
meta = pd.read_csv(DATA / "split_metadata.csv")
meta["name"] = meta["image"].map(lambda s: Path(s).name)
fold1 = meta[meta["fold"] == 1]
share = fold1["class_name"].value_counts() / len(fold1)
quota = (share * N_IMAGES).round().astype(int)
quota[quota.idxmax()] += N_IMAGES - quota.sum()   # make the quotas sum to N_IMAGES
rng = np.random.default_rng(SEED)
picked = []
for cls, n in quota.items():
    names = sorted(fold1.loc[fold1["class_name"] == cls, "name"])
    picked += list(rng.choice(names, size=n, replace=False))
BENCH = fold1[fold1["name"].isin(picked)].sort_values("name")[["name", "pid", "class_name", "fold"]]
bench_dir = WORK / "benchmark_images"
bench_dir.mkdir(exist_ok=True)
for n in BENCH["name"]:
    shutil.copy2(DATA / "val" / "images" / n, bench_dir / n)
BENCH.to_csv(WORK / "benchmark_images.csv", index=False)
(WORK / "benchmark_images.txt").write_text("\n".join(str(DATA / "val" / "images" / n) for n in BENCH["name"]))
print(BENCH["class_name"].value_counts().to_dict(), "| patients:", BENCH["pid"].nunique(), "| images:", len(BENCH))

## 7. CPU latency and memory (one fresh process per model and round, alternating order)

In [ ]:
%%writefile /kaggle/working/latency_worker.py
"""Time single-image end-to-end inference of one model on the Kaggle CPU.

End to end = Ultralytics predict() on an image path: image loading, letterbox
preprocessing, inference and NMS. Peak resident memory is sampled every 5 ms
from just before model loading to the end of the timed loop; the increase over
the pre-loading baseline is the memory attributable to the model.
"""

from __future__ import annotations

import argparse
import json
import threading
import time
from pathlib import Path

import psutil


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--model", required=True)
    parser.add_argument("--images", required=True, help="Text file, one image path per line.")
    parser.add_argument("--out", required=True)
    parser.add_argument("--threads", type=int, default=4)
    parser.add_argument("--warmup", type=int, default=10)
    parser.add_argument("--imgsz", type=int, default=640)
    parser.add_argument("--conf", type=float, default=0.25)
    parser.add_argument("--iou", type=float, default=0.7)
    parser.add_argument("--max-det", type=int, default=300)
    args = parser.parse_args()

    # Same number of interpreter threads as on the smartphone (Ultralytics leaves the default).
    import ai_edge_litert.interpreter as litert

    original_init = litert.Interpreter.__init__

    def init_with_threads(self, *a, **k):
        k.setdefault("num_threads", args.threads)
        original_init(self, *a, **k)

    litert.Interpreter.__init__ = init_with_threads

    import torch
    from ultralytics import YOLO

    torch.set_num_threads(args.threads)
    images = [line.strip() for line in Path(args.images).read_text().splitlines() if line.strip()]
    predict_args = dict(imgsz=args.imgsz, conf=args.conf, iou=args.iou, max_det=args.max_det,
                        device="cpu", verbose=False)

    process = psutil.Process()
    peak, stop = [process.memory_info().rss], threading.Event()

    def sample() -> None:
        while not stop.is_set():
            peak[0] = max(peak[0], process.memory_info().rss)
            time.sleep(0.005)

    baseline = process.memory_info().rss
    sampler = threading.Thread(target=sample, daemon=True)
    sampler.start()

    model = YOLO(args.model, task="detect")
    for path in images[: args.warmup]:  # discarded warm-up inferences
        model.predict(path, **predict_args)

    total_ms, stages, n_boxes = [], [], []
    for path in images:
        start = time.perf_counter()
        result = model.predict(path, **predict_args)[0]
        total_ms.append((time.perf_counter() - start) * 1000.0)
        stages.append(dict(result.speed))
        n_boxes.append(int(len(result.boxes)))

    stop.set()
    sampler.join()
    Path(args.out).write_text(json.dumps({
        "model": args.model,
        "threads": args.threads,
        "warmup": args.warmup,
        "images": images,
        "total_ms": total_ms,
        "stages_ms": stages,
        "boxes_per_image": n_boxes,
        "rss_baseline_mib": baseline / 2**20,
        "rss_peak_mib": peak[0] / 2**20,
        "rss_increase_mib": (peak[0] - baseline) / 2**20,
    }, indent=1))


if __name__ == "__main__":
    main()

In [ ]:
order = ["adopted_tflite", "yolov12n_tflite", "adopted_aug07_tflite"]
runs = []
for rnd in range(1, N_ROUNDS + 1):
    for key in (order if rnd % 2 else order[::-1]):
        out = WORK / "latency" / f"{key}_round{rnd}.json"
        out.parent.mkdir(exist_ok=True)
        subprocess.run([sys.executable, str(WORK / "latency_worker.py"), "--model", str(MODELS[key]),
                        "--images", str(WORK / "benchmark_images.txt"), "--out", str(out),
                        "--threads", str(THREADS), "--warmup", str(N_WARMUP),
                        "--imgsz", str(PREDICT["imgsz"]), "--conf", str(PREDICT["conf"]),
                        "--iou", str(PREDICT["iou"]), "--max-det", str(PREDICT["max_det"])], check=True)
        r = json.loads(out.read_text())
        r.update(key=key, round=rnd)
        runs.append(r)
        print(f"round {rnd} {key:26s} mean {np.mean(r['total_ms']):7.2f} ms  memory +{r['rss_increase_mib']:.1f} MiB")


def stats(values):
    v = np.asarray(values, dtype=float)
    return {"mean": v.mean(), "sd": v.std(ddof=1), "median": np.median(v), "min": v.min(), "max": v.max(),
            "p95": np.percentile(v, 95), "images_per_s": 1000.0 / v.mean()}


rows = []
for key in order:
    rr = [r for r in runs if r["key"] == key]
    total = [t for r in rr for t in r["total_ms"]]
    row = {"model": key, **stats(total),
           "round_means": [round(float(np.mean(r["total_ms"])), 2) for r in rr],
           "memory_increase_MiB_median": float(np.median([r["rss_increase_mib"] for r in rr]))}
    for stage in ("preprocess", "inference", "postprocess"):
        row[f"{stage}_ms_mean"] = float(np.mean([s[stage] for r in rr for s in r["stages_ms"]]))
    rows.append(row)
LATENCY = pd.DataFrame(rows).set_index("model")
print(LATENCY.round(2).to_string())

## 8. Results file and package for the smartphone benchmark

In [ ]:
results = {"environment": ENV, "export_args": EXPORT_ARGS, "predict_args": PREDICT, "threads": THREADS,
           "warmup": N_WARMUP, "rounds": N_ROUNDS, "checkpoints": [d_adopted, d_yolo],
           "exports": EXPORTS.to_dict(orient="records"), "fidelity_fold1": FIDELITY.reset_index().to_dict(orient="records"),
           "cpu_latency": LATENCY.reset_index().to_dict(orient="records"),
           "benchmark_images": BENCH.to_dict(orient="records")}
(WORK / "r1_8_cpu_results.json").write_text(json.dumps(results, indent=1, default=str))
FIDELITY.round(4).to_csv(WORK / "r1_8_fidelity_fold1.csv")
LATENCY.round(3).to_csv(WORK / "r1_8_cpu_latency.csv")

package = WORK / "r1_8_phone_package.zip"
with zipfile.ZipFile(package, "w", zipfile.ZIP_DEFLATED) as zf:
    for key in ("adopted_tflite", "yolov12n_tflite", "adopted_aug07_tflite"):
        zf.write(MODELS[key], f"models/{key}.tflite")
    for p in sorted(bench_dir.glob("*.png")):
        zf.write(p, f"benchmark_images/{p.name}")
    zf.write(WORK / "benchmark_images.csv", "benchmark_images.csv")
print("Saved r1_8_cpu_results.json, r1_8_fidelity_fold1.csv, r1_8_cpu_latency.csv and",
      f"{package.name} ({package.stat().st_size / 2**20:.1f} MiB)")